# Intro

**Seminar by @Elad_Benjo**

Dynamic Factor Models (DFM) extend the basic idea of Principal Component Analysis (PCA). PCA extracts “static factors” — linear combinations of many variables that summarize most of the variation in the data at a given point in time.

DFM pursues the same goal of dimension reduction but adds a time-series structure: the latent factors are assumed to evolve dynamically (often autoregressively), and the model uses the Kalman filter to estimate them even with missing values or mixed frequencies.

In short:

PCA → a static snapshot of common variation across variables.

DFM → a dynamic movie, where latent factors evolve over time and explain the co-movements of macroeconomic indicators.

In [1]:
from google.colab import drive
import sys
import pandas as pd

drive.mount('/content/drive')
sys.path.append('/content/drive/MyDrive/gdp_nowcasting_seminar/src')

Mounted at /content/drive


## Papers

https://www.federalreserve.gov/econres/ifdp/files/ifdp1385.pdf

https://www.bde.es/f/webbde/SES/Secciones/Publicaciones/PublicacionesSeriadas/DocumentosTrabajo/14/Fich/dt1425e.pdf

BOI cDFM

https://kamakama.gov.il/media/typa1sbo/dp202207en.pdf

#PLS

In [2]:
monthly_df = pd.read_pickle('/content/drive/MyDrive/gdp_nowcasting_seminar/Data/pickles/EDA/merged_data.pkl')

In [24]:
# Calculate the percentage of missing values for each column
missing_percentages = monthly_df.isnull().mean()

# Identify columns with more than 20% missing values
columns_to_drop = missing_percentages[missing_percentages > 0.20].index.tolist()

print("Columns with more than 20% missing values:")
print(columns_to_drop)

Columns with more than 20% missing values:
['Unemployment rate (ages 25-64,  percentage pots) ', 'Employment rate (ages 25-64) ', 'Participation rate (ages 25-64,  percentage pots) ', 'Industrial Production Index (Total)', 'cons_trust', 'Credit card purchase value dices (total) (2002=100)', 'salaried jobs ', 'Real Salary ']


In [22]:
monthly_df.head(50)

,Total refunds from the Income Tax Department,self employed returns,Cancellations Deductions,Capital Gas Tax Refunds,Cancellation companies,Purchase returns,Independent Cancellations,VAT refund autonomy and traders,"Unemployment rate (ages 25-64, percentage pots)",VAT_rate,...,tax differential companies,Companies advances,Income tax for self-employed dividuals and companies (advances and deductions),Deduction from salary,Deductions and the capital market,Total Gross Income Tax Division,TA125,SP500,USD_ILS,SPGSCI
Date,,,,,,,,,,,,,,,,,,,,,
1995-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1995-02-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,-0.183066,0.035439,-0.000908,0.006287
1995-03-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0.132758,0.026962,-0.010772,0.010950
1995-04-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0.038155,0.027577,-0.005832,0.025202
1995-05-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0.045318,0.035668,0.015208,-0.015476
1995-06-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0.036809,0.021055,-0.008140,-0.035579
1995-07-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0.015835,0.031282,-0.008377,0.016201
1995-08-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0.067450,-0.000320,0.025440,0.017828
1995-10-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,-0.048093,0.034323,-0.007383,0.014245


In [18]:
import pandas as pd
import numpy as np

# נניח ש־monthly_df הוא DataFrame עם DatetimeIndex (כמו אצלך)
threshold = 0.83  # 85% כיסוי לפחות

# חישוב שיעור התצפיות הלא-חסרות בכל חודש (אינדקס)
coverage_by_row = monthly_df.notna().mean(axis=1)

# יצירת מסכה של חודשים שעוברים את הסף
mask = coverage_by_row >= threshold

# איתור התאריך הראשון והאחרון ברצף שבו מתקיים הכיסוי
if mask.any():
    # Corrected: Use the Timestamp objects returned by idxmax() for indexing
    start_date = mask.idxmax()  # The first date where mask is True
    end_date = mask[::-1].idxmax()  # The last date where mask is True

    print(f"✅ טווח עם לפחות {threshold*100:.0f}% כיסוי: {start_date.date()} עד {end_date.date()}")
else:
    print(f"⚠️ אין אף חודש שבו כל העמודות עוברות {threshold*100:.0f}% כיסוי")

✅ טווח עם לפחות 83% כיסוי: 1998-03-01 עד 2025-02-01


## test-train split

In [ ]:
train_end = '2022-01-01'
test_start = '2022-02-01'

In [ ]:
x_train = df_X[:'2022-01-16']
x_test = df_X['2022-01-17':]

## Unsupervised

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.cross_decomposition import PLSRegression

In [33]:
# ---- Pre-imputation before PLS ----
monthly_df_filled = monthly_df.copy()

# 1. Remove columns that are mostly missing
drop_threshold = 0.8  # keep only columns with >= 80% non-missing
keep_cols = [c for c in monthly_df_filled.columns if monthly_df_filled[c].notna().mean() >= drop_threshold]
monthly_df_filled = monthly_df_filled[keep_cols]

# 2. Forward-fill and back-fill across time (for each column)
monthly_df_filled = monthly_df_filled.ffill().bfill()

# 3. Verify completeness
assert not monthly_df_filled.isna().any().any(), "Still missing values after imputation!"

# Now monthly_df_filled is full → safe to continue with PLS


In [34]:
monthly_df = monthly_df_filled
# ואז מריץ את הקוד של ה־PLS מהשלב הקודם


In [35]:
# -------- CONFIG --------
proxy_name = "The combed dex of the state of the economy"
max_ncomp  = 8
na_frac_threshold = 0.5
n_splits_ts = 10

# -------- 0) Ensure we have a `date` column from the DatetimeIndex --------
assert isinstance(monthly_df.index, pd.DatetimeIndex), "monthly_df must have a DatetimeIndex"
df = monthly_df.copy()

# Normalize date to first of monyh
date_col = df.index.to_period("M").to_timestamp("M")
df.insert(0, "date", date_col)
df = df.sort_values("date").reset_index(drop=True)

# -------- 1) Filter variables & build training set (X,y) without leakage --------
vars_all = [c for c in df.columns if c != "date"]
non_empty = [c for c in vars_all if df[c].notna().sum() > 35]
df = df[["date"] + non_empty]

assert proxy_name in df.columns, f"proxy_name '{proxy_name}' not found"
X_cols = [c for c in non_empty if c != proxy_name]

# שורה “כשרה” לאימון: proxy קיים וגם <=10% חוסרים ב-X
na_frac = 1.0 - df[X_cols].notna().mean(axis=1)
train_df = df.loc[(df[proxy_name].notna()) & (na_frac <= na_frac_threshold)].dropna(subset=X_cols + [proxy_name])

X_train = train_df[X_cols].to_numpy(float)
y_train = train_df[proxy_name].to_numpy(float).ravel()

max_ncomp_eff = int(min(max_ncomp, X_train.shape[1], len(train_df) - 2))
assert max_ncomp_eff >= 1, "Not enough rows/variables for PLS"

# -------- 2) Time-series CV to pick n_components (no leakage) --------
def cv_rmse_for_ncomp(n_comp: int) -> float:
    tscv = TimeSeriesSplit(n_splits=n_splits_ts)
    rmses = []
    for tr_idx, va_idx in tscv.split(X_train):
        X_tr, X_va = X_train[tr_idx], X_train[va_idx]
        y_tr, y_va = y_train[tr_idx], y_train[va_idx]

        xsc, ysc = StandardScaler(), StandardScaler()
        X_tr_s = xsc.fit_transform(X_tr)
        y_tr_s = ysc.fit_transform(y_tr.reshape(-1, 1)).ravel()

        pls = PLSRegression(n_components=n_comp, scale=False)
        pls.fit(X_tr_s, y_tr_s)

        y_hat_s = pls.predict(xsc.transform(X_va)).ravel()
        y_hat = ysc.inverse_transform(y_hat_s.reshape(-1, 1)).ravel()
        rmses.append(np.sqrt(mean_squared_error(y_va, y_hat)))
    return float(np.mean(rmses))

rmse_grid = [(k, cv_rmse_for_ncomp(k)) for k in range(1, max_ncomp_eff + 1)]
best_ncomp = min(rmse_grid, key=lambda t: t[1])[0]
print(f"[PLS] Best n_components={best_ncomp} | grid={rmse_grid}")

# -------- 3) Fit final PLS on full training sample --------
xsc_final, ysc_final = StandardScaler(), StandardScaler()
X_train_s = xsc_final.fit_transform(X_train)
y_train_s = ysc_final.fit_transform(y_train.reshape(-1, 1)).ravel()

pls_final = PLSRegression(n_components=best_ncomp, scale=False)
pls_final.fit(X_train_s, y_train_s)

# גורמים לחלק האימון (לא חובה, שימושי לבקרה)
scores_train = pd.DataFrame(
    pls_final.x_scores_,
    columns=[f"f_pls{i}" for i in range(1, best_ncomp + 1)],
    index=train_df.index
)
factors_train = pd.concat([train_df[["date"]].reset_index(drop=True),
                           scores_train.reset_index(drop=True)], axis=1)

# -------- 4) Score PLS factors for the FULL monthly panel (ragged-edge aware) --------
panel = df[["date"] + X_cols].copy()
panel["na_frac"] = 1.0 - panel[X_cols].notna().mean(axis=1)
pred_ready = panel.loc[panel["na_frac"] <= na_frac_threshold].drop(columns="na_frac").dropna()

X_pred_s = xsc_final.transform(pred_ready[X_cols].to_numpy(float))
scores_pred = pls_final.transform(X_pred_s)  # x-scores for new data
scores_pred = pd.DataFrame(scores_pred, columns=[f"f_pls{i}" for i in range(1, best_ncomp + 1)])
factors_pred = pd.concat([pred_ready[["date"]].reset_index(drop=True), scores_pred], axis=1)

# Normalize date to first of month in factors_pred
factors_pred = factors_pred.set_index("date")
factors_pred.index = factors_pred.index.to_period("M").to_timestamp("M")
factors_pred = factors_pred.reset_index()


# אינדקס חודשי מלא מהאינדקס המקורי
monthly_index = pd.DataFrame({"date": pd.date_range(df["date"].min(), df["date"].max(), freq="MS")})

# Print statements to inspect the dataframes before merging
print("monthly_index head:")
print(monthly_index.head())
print("\nfactors_pred head:")
print(factors_pred.head())

pls_factors_monthly = monthly_index.merge(factors_pred, on="date", how="left").sort_values("date")

# גם ה-proxy מיושר לאינדקס מלא
monthly_proxy = monthly_index.merge(
    df[["date", proxy_name]].rename(columns={proxy_name: "proxy"}),
    on="date", how="left"
).sort_values("date")

# -------- 5) Quick checks --------
print(pls_factors_monthly.head())
print(pls_factors_monthly.tail())
print(monthly_proxy.tail())

/usr/local/lib/python3.12/dist-packages/sklearn/cross_decomposition/_pls.py:348: UserWarning: y residual is constant at iteration 0
  warnings.warn(f"y residual is constant at iteration {k}")
/usr/local/lib/python3.12/dist-packages/sklearn/cross_decomposition/_pls.py:348: UserWarning: y residual is constant at iteration 0
  warnings.warn(f"y residual is constant at iteration {k}")
/usr/local/lib/python3.12/dist-packages/sklearn/cross_decomposition/_pls.py:348: UserWarning: y residual is constant at iteration 0
  warnings.warn(f"y residual is constant at iteration {k}")
/usr/local/lib/python3.12/dist-packages/sklearn/cross_decomposition/_pls.py:348: UserWarning: y residual is constant at iteration 0
  warnings.warn(f"y residual is constant at iteration {k}")
/usr/local/lib/python3.12/dist-packages/sklearn/cross_decomposition/_pls.py:348: UserWarning: y residual is constant at iteration 0
  warnings.warn(f"y residual is constant at iteration {k}")
/usr/local/lib/python3.12/dist-packages/

[PLS] Best n_components=2 | grid=[(1, 0.00426711926052937), (2, 0.004257613127647867), (3, 0.0043876561929878525), (4, 0.004455774949167772), (5, 0.004553605143757219), (6, 0.004612702201663192), (7, 0.004672291613753622), (8, 0.004737726001139823)]
monthly_index head:
        date
0 1995-02-01
1 1995-03-01
2 1995-04-01
3 1995-05-01
4 1995-06-01

factors_pred head:
        date    f_pls1    f_pls2
0 1995-01-31 -0.917998  0.219144
1 1995-02-28 -0.917998  0.219144
2 1995-03-31  0.251917  0.291780
3 1995-04-30 -0.030922  0.556485
4 1995-05-31 -0.481789  0.378443
        date  f_pls1  f_pls2
0 1995-02-01     NaN     NaN
1 1995-03-01     NaN     NaN
2 1995-04-01     NaN     NaN
3 1995-05-01     NaN     NaN
4 1995-06-01     NaN     NaN
          date  f_pls1  f_pls2
358 2024-12-01     NaN     NaN
359 2025-01-01     NaN     NaN
360 2025-02-01     NaN     NaN
361 2025-03-01     NaN     NaN
362 2025-04-01     NaN     NaN
          date  proxy
358 2024-12-01    NaN
359 2025-01-01    NaN
360 2025

/usr/local/lib/python3.12/dist-packages/sklearn/cross_decomposition/_pls.py:348: UserWarning: y residual is constant at iteration 0
  warnings.warn(f"y residual is constant at iteration {k}")
/usr/local/lib/python3.12/dist-packages/sklearn/cross_decomposition/_pls.py:348: UserWarning: y residual is constant at iteration 0
  warnings.warn(f"y residual is constant at iteration {k}")


In [29]:
factors_pred.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129 entries, 0 to 128
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    129 non-null    datetime64[ns]
 1   f_pls1  129 non-null    float64       
 2   f_pls2  129 non-null    float64       
dtypes: datetime64[ns](1), float64(2)
memory usage: 3.2 KB


In [36]:
print(factors_pred.head())
print(factors_pred.tail())

        date    f_pls1    f_pls2
0 1995-01-31 -0.917998  0.219144
1 1995-02-28 -0.917998  0.219144
2 1995-03-31  0.251917  0.291780
3 1995-04-30 -0.030922  0.556485
4 1995-05-31 -0.481789  0.378443
          date    f_pls1    f_pls2
303 2024-12-31  2.664379  0.754443
304 2025-01-31 -2.130236 -0.265092
305 2025-02-28 -0.026900  2.020714
306 2025-03-31 -0.842725 -0.193933
307 2025-04-30  0.012751 -1.780346


## Supervised